# Treinamento com interface de alto nível

## Importação das bibliotecas

In [172]:
# http://pytorch.org/
from os.path import exists

import torch

In [173]:
import argparse
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import Dataset, random_split
from torch.optim.lr_scheduler import StepLR

import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split

# Dataset

In [174]:
#!/bin/bash
!curl -L -o /content/old-car-price-prediction.zip https://www.kaggle.com/api/v1/datasets/download/milanvaddoriya/old-car-price-prediction

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  105k  100  105k    0     0   155k      0 --:--:-- --:--:-- --:--:--  155k


In [175]:
!unzip -o /content/old-car-price-prediction.zip -d /content/old-car-price-prediction

Archive:  /content/old-car-price-prediction.zip
  inflating: /content/old-car-price-prediction/car_price.csv  


In [176]:
old_car_prediction_csv = "/content/old-car-price-prediction/car_price.csv"

In [177]:
df = pd.read_csv(old_car_prediction_csv)
df.head()

,Unnamed: 0,car_name,car_prices_in_rupee,kms_driven,fuel_type,transmission,ownership,manufacture,engine,Seats
0,0,Jeep Compass 2.0 Longitude Option BSIV,10.03 Lakh,"86,226 kms",Diesel,Manual,1st Owner,2017,1956 cc,5 Seats
1,1,Renault Duster RXZ Turbo CVT,12.83 Lakh,"13,248 kms",Petrol,Automatic,1st Owner,2021,1330 cc,5 Seats
2,2,Toyota Camry 2.5 G,16.40 Lakh,"60,343 kms",Petrol,Automatic,1st Owner,2016,2494 cc,5 Seats
3,3,Honda Jazz VX CVT,7.77 Lakh,"26,696 kms",Petrol,Automatic,1st Owner,2018,1199 cc,5 Seats
4,4,Volkswagen Polo 1.2 MPI Highline,5.15 Lakh,"69,414 kms",Petrol,Manual,1st Owner,2016,1199 cc,5 Seats


In [178]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5512 entries, 0 to 5511
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Unnamed: 0           5512 non-null   int64 
 1   car_name             5512 non-null   object
 2   car_prices_in_rupee  5512 non-null   object
 3   kms_driven           5512 non-null   object
 4   fuel_type            5512 non-null   object
 5   transmission         5512 non-null   object
 6   ownership            5512 non-null   object
 7   manufacture          5512 non-null   int64 
 8   engine               5512 non-null   object
 9   Seats                5512 non-null   object
dtypes: int64(2), object(8)
memory usage: 430.8+ KB


In [179]:
df.isnull().sum()

,0
Unnamed: 0,0
car_name,0
car_prices_in_rupee,0
kms_driven,0
fuel_type,0
transmission,0
ownership,0
manufacture,0
engine,0
Seats,0


In [180]:
df.duplicated().sum()

np.int64(0)

In [181]:
df.dtypes

,0
Unnamed: 0,int64
car_name,object
car_prices_in_rupee,object
kms_driven,object
fuel_type,object
transmission,object
ownership,object
manufacture,int64
engine,object
Seats,object


In [182]:
df.nunique()

,0
Unnamed: 0,5512
car_name,1896
car_prices_in_rupee,1300
kms_driven,2610
fuel_type,5
transmission,2
ownership,6
manufacture,26
engine,139
Seats,6


In [183]:
df.describe()

,Unnamed: 0,manufacture
count,5512.000000,5512.000000
mean,2755.500000,2015.455552
std,1591.321673,3.927974
min,0.000000,1995.000000
25%,1377.750000,2013.000000
50%,2755.500000,2016.000000
75%,4133.250000,2018.000000
max,5511.000000,2022.000000


In [184]:
def car_model(x):
  return x[x.index(" ") + 1:]
def car_marca(x):
  return x[:x.index(" ")]
df["marca"] = df["car_name"].apply(car_marca)
df["car_name"] = df["car_name"].apply(car_model)


In [185]:
df.head()

,Unnamed: 0,car_name,car_prices_in_rupee,kms_driven,fuel_type,transmission,ownership,manufacture,engine,Seats,marca
0,0,Compass 2.0 Longitude Option BSIV,10.03 Lakh,"86,226 kms",Diesel,Manual,1st Owner,2017,1956 cc,5 Seats,Jeep
1,1,Duster RXZ Turbo CVT,12.83 Lakh,"13,248 kms",Petrol,Automatic,1st Owner,2021,1330 cc,5 Seats,Renault
2,2,Camry 2.5 G,16.40 Lakh,"60,343 kms",Petrol,Automatic,1st Owner,2016,2494 cc,5 Seats,Toyota
3,3,Jazz VX CVT,7.77 Lakh,"26,696 kms",Petrol,Automatic,1st Owner,2018,1199 cc,5 Seats,Honda
4,4,Polo 1.2 MPI Highline,5.15 Lakh,"69,414 kms",Petrol,Manual,1st Owner,2016,1199 cc,5 Seats,Volkswagen


In [186]:
df[['price', 'multiply']] = df['car_prices_in_rupee'].str.split(' ', expand=True)
df.drop('car_prices_in_rupee', axis=1, inplace=True)

In [187]:
df.head()

,Unnamed: 0,car_name,kms_driven,fuel_type,transmission,ownership,manufacture,engine,Seats,marca,price,multiply
0,0,Compass 2.0 Longitude Option BSIV,"86,226 kms",Diesel,Manual,1st Owner,2017,1956 cc,5 Seats,Jeep,10.03,Lakh
1,1,Duster RXZ Turbo CVT,"13,248 kms",Petrol,Automatic,1st Owner,2021,1330 cc,5 Seats,Renault,12.83,Lakh
2,2,Camry 2.5 G,"60,343 kms",Petrol,Automatic,1st Owner,2016,2494 cc,5 Seats,Toyota,16.40,Lakh
3,3,Jazz VX CVT,"26,696 kms",Petrol,Automatic,1st Owner,2018,1199 cc,5 Seats,Honda,7.77,Lakh
4,4,Polo 1.2 MPI Highline,"69,414 kms",Petrol,Manual,1st Owner,2016,1199 cc,5 Seats,Volkswagen,5.15,Lakh


In [188]:
# Var: kms_driven - removing kms and convert to int
df['kms_driven'] = df['kms_driven'].str.replace(' kms', '')
df['kms_driven'] = df['kms_driven'].str.replace(',', '').astype('int')

# Var: manufacture -  convert to int
df['manufacture'] = df['manufacture'].astype('int')

# Var: engine - remove cc and convert to int
df['engine'] = df['engine'].str.replace(' cc', '').astype('int')

#Var: Seats - remove seats and convert to int
df['Seats'] = df['Seats'].str.replace(' Seats', '').astype('int')

# Var price - convert to float:
df['price'] = df['price'].str.replace(',', '.').astype('float')

In [189]:
df.head()

,Unnamed: 0,car_name,kms_driven,fuel_type,transmission,ownership,manufacture,engine,Seats,marca,price,multiply
0,0,Compass 2.0 Longitude Option BSIV,86226,Diesel,Manual,1st Owner,2017,1956,5,Jeep,10.03,Lakh
1,1,Duster RXZ Turbo CVT,13248,Petrol,Automatic,1st Owner,2021,1330,5,Renault,12.83,Lakh
2,2,Camry 2.5 G,60343,Petrol,Automatic,1st Owner,2016,2494,5,Toyota,16.40,Lakh
3,3,Jazz VX CVT,26696,Petrol,Automatic,1st Owner,2018,1199,5,Honda,7.77,Lakh
4,4,Polo 1.2 MPI Highline,69414,Petrol,Manual,1st Owner,2016,1199,5,Volkswagen,5.15,Lakh


In [190]:
df['price']=np.where(df['multiply'] == 'Crore',
                                           df['price'] * 100,
                                           df['price'])
df['price']=df['price']*100000

df.drop('multiply', axis=1, inplace=True)

In [191]:
df.head()

,Unnamed: 0,car_name,kms_driven,fuel_type,transmission,ownership,manufacture,engine,Seats,marca,price
0,0,Compass 2.0 Longitude Option BSIV,86226,Diesel,Manual,1st Owner,2017,1956,5,Jeep,1003000.0
1,1,Duster RXZ Turbo CVT,13248,Petrol,Automatic,1st Owner,2021,1330,5,Renault,1283000.0
2,2,Camry 2.5 G,60343,Petrol,Automatic,1st Owner,2016,2494,5,Toyota,1640000.0
3,3,Jazz VX CVT,26696,Petrol,Automatic,1st Owner,2018,1199,5,Honda,777000.0
4,4,Polo 1.2 MPI Highline,69414,Petrol,Manual,1st Owner,2016,1199,5,Volkswagen,515000.0


In [192]:
def owner(x):
  return x[:1]
df["ownership"] = df["ownership"].apply(owner)
df.head()

,Unnamed: 0,car_name,kms_driven,fuel_type,transmission,ownership,manufacture,engine,Seats,marca,price
0,0,Compass 2.0 Longitude Option BSIV,86226,Diesel,Manual,1,2017,1956,5,Jeep,1003000.0
1,1,Duster RXZ Turbo CVT,13248,Petrol,Automatic,1,2021,1330,5,Renault,1283000.0
2,2,Camry 2.5 G,60343,Petrol,Automatic,1,2016,2494,5,Toyota,1640000.0
3,3,Jazz VX CVT,26696,Petrol,Automatic,1,2018,1199,5,Honda,777000.0
4,4,Polo 1.2 MPI Highline,69414,Petrol,Manual,1,2016,1199,5,Volkswagen,515000.0


In [193]:
categorical_columns = ["car_name", "fuel_type", "transmission", "marca"]
for col in categorical_columns:
  print(f"Categoria na {col}: {df[col].unique()}")

Categoria na car_name: ['Compass 2.0 Longitude Option BSIV' 'Duster RXZ Turbo CVT' 'Camry 2.5 G'
 ... 'XC 90 D5 Momentum BSIV' 'E-Class E250 Edition E' 'M Series M4 Coupe']
Categoria na fuel_type: ['Diesel' 'Petrol' 'Cng' 'Electric' 'Lpg']
Categoria na transmission: ['Manual' 'Automatic']
Categoria na marca: ['Jeep' 'Renault' 'Toyota' 'Honda' 'Volkswagen' 'Maruti' 'Mahindra'
 'Hyundai' 'Nissan' 'Kia' 'MG' 'Tata' 'BMW' 'Mercedes-Benz' 'Datsun'
 'Volvo' 'Audi' 'Porsche' 'Ford' 'Chevrolet' 'Skoda' 'Lexus' 'Land' 'Mini'
 'Jaguar' 'Mitsubishi' 'Force' 'Premier' 'Fiat' 'Maserati' 'Bentley'
 'Isuzu']


In [194]:
df["ownership"] = df["ownership"].astype('int')

In [195]:
df.describe()

,Unnamed: 0,kms_driven,ownership,manufacture,engine,Seats,price
count,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5512.000000,5.512000e+03
mean,2755.500000,63211.888062,1.421807,2015.455552,1532.299710,5.250726,1.329907e+06
std,1591.321673,41844.131167,0.703092,3.927974,579.210876,0.720075,2.192714e+06
min,0.000000,250.000000,0.000000,1995.000000,0.000000,2.000000,1.000000e+05
25%,1377.750000,33151.750000,1.000000,2013.000000,1197.000000,5.000000,3.300000e+05
50%,2755.500000,59000.000000,1.000000,2016.000000,1396.000000,5.000000,5.750000e+05
75%,4133.250000,84265.250000,2.000000,2018.000000,1950.000000,5.000000,1.149250e+06
max,5511.000000,560000.000000,5.000000,2022.000000,5950.000000,8.000000,1.920000e+07


In [196]:
car_name_encoded = LabelEncoder()
df["car_name"] = car_name_encoded.fit_transform(df["car_name"])

fuel_type_encoded = LabelEncoder()
df["fuel_type"] = fuel_type_encoded.fit_transform(df["fuel_type"])

transmission_encoded = LabelEncoder()
df["transmission"] = transmission_encoded.fit_transform(df["transmission"])

marca_encoded = LabelEncoder()
df["marca"] = marca_encoded.fit_transform(df["marca"])


In [197]:
df.head()

,Unnamed: 0,car_name,kms_driven,fuel_type,transmission,ownership,manufacture,engine,Seats,marca,price
0,0,426,86226,1,1,1,2017,1956,5,12,1003000.0
1,1,509,13248,4,0,1,2021,1330,5,26,1283000.0
2,2,290,60343,4,0,1,2016,2494,5,29,1640000.0
3,3,912,26696,4,0,1,2018,1199,5,8,777000.0
4,4,1095,69414,4,1,1,2016,1199,5,30,515000.0


In [198]:
standart_scaler = StandardScaler()
new_df = pd.DataFrame(standart_scaler.fit_transform(df), columns=df.columns)

In [199]:
new_df.head()

,Unnamed: 0,car_name,kms_driven,fuel_type,transmission,ownership,manufacture,engine,Seats,marca,price
0,-1.731737,-0.915502,0.550046,-1.068243,0.625473,-0.599986,0.393228,0.731579,-0.348225,-0.425853,-0.149101
1,-1.731108,-0.771939,-1.194156,0.918538,-1.598790,-0.599986,1.411657,-0.349299,-0.348225,1.296126,-0.021394
2,-1.730480,-1.150738,-0.068568,0.918538,-1.598790,-0.599986,0.138621,1.660514,-0.348225,1.665122,0.141433
3,-1.729851,-0.074879,-0.872744,0.918538,-1.598790,-0.599986,0.647835,-0.575490,-0.348225,-0.917847,-0.252179
4,-1.729223,0.241652,0.148233,0.918538,0.625473,-0.599986,0.138621,-0.575490,-0.348225,1.788120,-0.371677


In [200]:
x = new_df[["car_name","kms_driven","fuel_type","transmission","ownership","manufacture","engine","Seats","marca"]]
y = new_df["price"]

In [201]:
y = y.values.reshape(-1, 1)
x = x.to_numpy()

In [202]:
x

array([[-0.91550178,  0.55004607, -1.0682431 , ...,  0.73157937,
        -0.34822548, -0.42585338],
       [-0.77193855, -1.19415602,  0.91853768, ..., -0.34929949,
        -0.34822548,  1.29612594],
       [-1.15073792, -0.06856752,  0.91853768, ...,  1.6605136 ,
        -0.34822548,  1.66512151],
       ...,
       [ 1.31059313,  0.64024611, -1.0682431 , ...,  2.11116757,
        -0.34822548, -0.5488519 ],
       [-1.60910245,  0.37734191, -1.0682431 , ...,  2.477216  ,
         1.04064487, -1.77883713],
       [-1.62985858, -0.50697223, -1.0682431 , ...,  0.7920119 ,
        -0.34822548, -1.77883713]])

In [203]:
y

array([[-0.14910131],
       [-0.02139412],
       [ 0.14143255],
       ...,
       [ 0.02056682],
       [ 0.75716366],
       [ 0.84838308]])

In [204]:
class OldCarDataset(Dataset):
  def __init__(self, x, y):
    super(OldCarDataset, self).__init__()
    if(len(x) != len(y)):
      raise ValueError("Size de X e Y não dão match!")
    self.x = torch.FloatTensor(x)
    self.y = torch.FloatTensor(y)
    self.x = torch.nan_to_num(self.x)
    self.y = torch.nan_to_num(self.y)

  def __getitem__(self, index):
     return self.x[index], self.y[index]

  def __len__(self):
    return len(self.x)

In [205]:
dataset = OldCarDataset(x, y)

In [206]:
train_data, val_data = random_split(dataset, [0.7, 0.3])

In [207]:
print(f"len treinamento: {len(train_data)}")

len treinamento: 3859


In [208]:
print(f"len val: {len(val_data)}")

len val: 1653


In [209]:
x.shape

(5512, 9)

## Criação da rede

In [210]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.layer1 = nn.Linear(9, 32)
        self.layer2 = nn.Linear(32, 64)
        self.layer2_1 = nn.Linear(64, 128)
        self.layer2_2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 32)
        self.layer4 = nn.Linear(32, 1)

    def forward(self, x):
        x = self.layer1(x)
        x = F.relu(x)
        x = self.layer2(x)
        x = F.relu(x)
        x = self.layer2_1(x)
        x = F.relu(x)
        x = self.layer2_2(x)
        x = F.relu(x)
        x = self.layer3(x)
        x = F.relu(x)
        output = self.layer4(x)
        return output

model = Net()

In [211]:
model

Net(
  (layer1): Linear(in_features=9, out_features=32, bias=True)
  (layer2): Linear(in_features=32, out_features=64, bias=True)
  (layer2_1): Linear(in_features=64, out_features=128, bias=True)
  (layer2_2): Linear(in_features=128, out_features=64, bias=True)
  (layer3): Linear(in_features=64, out_features=32, bias=True)
  (layer4): Linear(in_features=32, out_features=1, bias=True)
)

## Treinamento

### Criando o objeto de treinamento

In [212]:
def train(log_interval, dry_run, model, device, train_loader, optimizer, epoch, criterion):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % log_interval == 0:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
                epoch, batch_idx * len(data), len(train_loader.dataset),
                100. * batch_idx / len(train_loader), loss.item()))
            if dry_run:
                break

In [213]:
def test(model, device, test_loader, criterion):
    model.eval()
    total_loss = 0
    total_samples = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output, target)
            total_loss += loss.item() * target.size(0)
            total_samples += target.size(0)
        mse = total_loss / total_samples
        print(f"\nMSE validation: {mse:.4f}\n")
    return mse

## Avaliação

In [217]:
use_cuda = torch.cuda.is_available()

torch.manual_seed(1111)

device = torch.device("cuda" if use_cuda else "cpu")

train_kwargs = {'batch_size': 300}
test_kwargs = {'batch_size': 1000}
if use_cuda:
    cuda_kwargs = {'num_workers': 1,
                    'pin_memory': True,
                    'shuffle': True}
    train_kwargs.update(cuda_kwargs)
    test_kwargs.update(cuda_kwargs)

train_loader = torch.utils.data.DataLoader(train_data,**train_kwargs)
test_loader = torch.utils.data.DataLoader(val_data, **test_kwargs)
model = Net().to(device)
optimizer = optim.SGD(model.parameters(), lr=0.01)
criterion = nn.MSELoss()
best_loss = test(model, device, test_loader, criterion)
epochs = 300
# scheduler = StepLR(optimizer, step_size=1, gamma=0.7)

for epoch in range(1, epochs + 1):
    train(10, False, model, device, train_loader, optimizer, epoch, criterion)
    loss = test(model, device, test_loader, criterion)
    if (loss < best_loss):
      best_loss = loss
      torch.save(model.state_dict(), "old_car_nn.pt")


MSE validation: 1.1375

Train Epoch: 1 [0/3859 (0%)]	Loss: 1.203932
Train Epoch: 1 [3000/3859 (77%)]	Loss: 0.801526

MSE validation: 1.1370

Train Epoch: 2 [0/3859 (0%)]	Loss: 0.886171
Train Epoch: 2 [3000/3859 (77%)]	Loss: 0.703734

MSE validation: 1.1367

Train Epoch: 3 [0/3859 (0%)]	Loss: 0.775273
Train Epoch: 3 [3000/3859 (77%)]	Loss: 0.773928

MSE validation: 1.1365

Train Epoch: 4 [0/3859 (0%)]	Loss: 0.819317
Train Epoch: 4 [3000/3859 (77%)]	Loss: 0.654363

MSE validation: 1.1363

Train Epoch: 5 [0/3859 (0%)]	Loss: 1.001948
Train Epoch: 5 [3000/3859 (77%)]	Loss: 0.715161

MSE validation: 1.1362

Train Epoch: 6 [0/3859 (0%)]	Loss: 0.879718
Train Epoch: 6 [3000/3859 (77%)]	Loss: 0.565711

MSE validation: 1.1360

Train Epoch: 7 [0/3859 (0%)]	Loss: 1.036060
Train Epoch: 7 [3000/3859 (77%)]	Loss: 0.747147

MSE validation: 1.1358

Train Epoch: 8 [0/3859 (0%)]	Loss: 0.575030
Train Epoch: 8 [3000/3859 (77%)]	Loss: 0.780354

MSE validation: 1.1356

Train Epoch: 9 [0/3859 (0%)]	Loss: 0.77